In [1]:
# ============================================
# Standard library imports
# ============================================

import sys
import re
import logging
import warnings
from pathlib import Path

from IPython.display import display

warnings.filterwarnings("ignore")


# ============================================
# Numeric / statistics
# ============================================

import numpy as np
import pandas as pd

import scipy
from scipy import stats
from scipy.stats import spearmanr, pearsonr


# ============================================
# Plotting / visualization
# ============================================

import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter
from matplotlib.colors import LinearSegmentedColormap

import seaborn as sns


# ============================================
# Version info
# ============================================

print(f"python  = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas  = {pd.__version__}")
print(f"numpy   = {np.__version__}")
print(f"scipy   = {scipy.__version__}")


# ============================================
# Logging
# ============================================

logging.root.handlers = []

stream_handler = logging.StreamHandler(sys.stderr)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)

logger = logging.getLogger(__name__)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python  = 3.11.15
pandas  = 2.3.3
numpy   = 2.4.6
scipy   = 1.17.1


In [2]:
# ============================================
# Analysis configuration
# ============================================

RESULT_DIR = Path("../results")
FIGURE_DIR = RESULT_DIR / "figures"
TABLE_DIR = RESULT_DIR / "table"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_LIST = ["AT21", "NB21", "OS21"]

MODEL_INFO = {
    "lgbm": {
        "label": "LightGBM",
        "dir": RESULT_DIR / "lgbm",
        "color": "#664D0AFF",
    },
    "rf": {
        "label": "Random Forest",
        "dir": RESULT_DIR / "rf",
        "color": "#13563f",
    },
}

DATASET_ORDER = ["Training", "Test"]

print("RESULT_DIR:", RESULT_DIR.resolve())
print("FIGURE_DIR:", FIGURE_DIR.resolve())
print("TABLE_DIR :", TABLE_DIR.resolve())

RESULT_DIR: /home/ha-ibnu/Code/regression/results
FIGURE_DIR: /home/ha-ibnu/Code/regression/results/figures
TABLE_DIR : /home/ha-ibnu/Code/regression/results/table


In [3]:
# ============================================
# Plotting configuration & helpers
# ============================================

_PLOT_CFG = {
    "fig_w": 5.5,
    "fig_h": 4.5,
    "dpi": 300,
}


def set_plot_style(
    *,
    font_size=12,
    dpi=300,
    axes_linewidth=1.2,
    spines_top=True,
    spines_right=True,
    tick_size_major=5,
    tick_dir="out",
    grid=False,
):
    sns.set_style("ticks")

    mpl.rcParams.update({
        "font.size": font_size,
        "figure.dpi": dpi,
        "axes.linewidth": axes_linewidth,
        "axes.spines.top": spines_top,
        "axes.spines.right": spines_right,
        "xtick.major.size": tick_size_major,
        "ytick.major.size": tick_size_major,
        "xtick.direction": tick_dir,
        "ytick.direction": tick_dir,
        "axes.grid": grid,
        "figure.autolayout": False,
    })

    _PLOT_CFG["dpi"] = dpi


def make_fig(w=None, h=None, dpi=None):
    W = float(w) if w is not None else _PLOT_CFG["fig_w"]
    H = float(h) if h is not None else _PLOT_CFG["fig_h"]
    D = dpi if dpi is not None else _PLOT_CFG["dpi"]

    fig, ax = plt.subplots(figsize=(W, H), dpi=D)
    return fig, ax


def format_axis(ax, *, xlabel=None, ylabel=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    return ax


set_plot_style()

In [4]:
# ============================================
# Statistical helper functions
# ============================================

def safe_pearsonr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return pearsonr(x, y)


def safe_spearmanr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return spearmanr(x, y)


def summarize_numeric(df, group_cols, value_col):
    return (
        df
        .groupby(group_cols, dropna=False)[value_col]
        .agg(["count", "mean", "std", "median", "min", "max"])
        .reset_index()
    )

In [6]:
# ============================================
# Load saved Step 1 result tables
# ============================================

def load_result_table(model_key, suffix):
    model_dir = MODEL_INFO[model_key]["dir"]
    rows = []

    for species in SPECIES_LIST:
        file = model_dir / f"{species}.{model_key}.{suffix}.tsv"

        if not file.exists():
            logger.warning("Missing file: %s", file)
            continue

        df = pd.read_csv(file, sep="\t")
        rows.append(df)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


metrics_df = pd.concat([
    load_result_table("lgbm", "metrics"),
    load_result_table("rf", "metrics"),
], ignore_index=True)

best_params_df = pd.concat([
    load_result_table("lgbm", "best_params"),
    load_result_table("rf", "best_params"),
], ignore_index=True)

feature_importance_df = pd.concat([
    load_result_table("lgbm", "feature_importance"),
    load_result_table("rf", "feature_importance"),
], ignore_index=True)

shap_summary_df = pd.concat([
    load_result_table("lgbm", "shap_summary"),
    load_result_table("rf", "shap_summary"),
], ignore_index=True)

cv_region_df = pd.concat([
    load_result_table("lgbm", "cv_by_region"),
    load_result_table("rf", "cv_by_region"),
], ignore_index=True)

cv_feature_type_df = pd.concat([
    load_result_table("lgbm", "cv_by_feature_type"),
    load_result_table("rf", "cv_by_feature_type"),
], ignore_index=True)

prediction_df = pd.concat([
    load_result_table("lgbm", "predictions"),
    load_result_table("rf", "predictions"),
], ignore_index=True)

# ============================================
# Quick sanity check
# ============================================

tables = {
    "metrics": metrics_df,
    "best_params": best_params_df,
    "feature_importance": feature_importance_df,
    "shap_summary": shap_summary_df,
    "cv_region": cv_region_df,
    "cv_feature_type": cv_feature_type_df,
    "prediction": prediction_df,
}

for name, df in tables.items():
    print(f"{name:20s}: {df.shape}")

display(metrics_df.head())
display(prediction_df.head())

metrics             : (12, 22)
best_params         : (6, 14)
feature_importance  : (2156, 5)
shap_summary        : (2254, 7)
cv_region           : (90, 9)
cv_feature_type     : (1530, 9)
prediction          : (36118, 10)


,species,model,dataset,R2,Pearson_r,RMSE,MAE,best_cv_R2,n_train,n_test,...,k_cv,n_estimators,learning_rate,best_param.max_depth,best_param.min_child_samples,best_param.feature_fraction,best_param.num_leaves,best_param.min_samples_leaf,best_param.min_samples_split,best_param.max_features
0,AT21,LightGBM,Training,0.556015,0.771511,0.666322,0.511277,0.252106,5698,1428,...,10,100,0.05,7,30.0,0.3,58.0,NaN,NaN,NaN
1,AT21,LightGBM,Test,0.298263,0.548332,0.842763,0.643657,0.252106,5698,1428,...,10,100,0.05,7,30.0,0.3,58.0,NaN,NaN,NaN
2,NB21,LightGBM,Training,0.590997,0.789585,0.639533,0.490118,0.310320,4305,1076,...,10,100,0.05,7,10.0,0.3,31.0,NaN,NaN,NaN
3,NB21,LightGBM,Test,0.291868,0.542054,0.857198,0.649158,0.310320,4305,1076,...,10,100,0.05,7,10.0,0.3,31.0,NaN,NaN,NaN
4,OS21,LightGBM,Training,0.717678,0.871608,0.531340,0.408141,0.364618,4445,1107,...,10,100,0.05,7,3.0,0.5,56.0,NaN,NaN,NaN


,species,model,var_id,trans_id,gene_id,dataset,observed,predicted,residual,abs_residual
0,AT21,LightGBM,AT5G05370.1.1591901.1590815,AT5G05370.1,AT5G05370,Training,0.054841,0.466496,-0.411655,NaN
1,AT21,LightGBM,AT5G16060.1.5246121.5247413,AT5G16060.1,AT5G16060,Training,-1.094430,-0.364117,-0.730313,NaN
2,AT21,LightGBM,AT2G34160.1.14426246.14427367,AT2G34160.1,AT2G34160,Training,-0.073162,-0.028276,-0.044886,NaN
3,AT21,LightGBM,AT5G54600.1.22183004.22184509,AT5G54600.1,AT5G54600,Training,-0.302488,-0.468002,0.165514,NaN
4,AT21,LightGBM,AT2G23340.1.9937988.9938873,AT2G23340.1,AT2G23340,Training,1.491571,0.321837,1.169734,NaN


In [7]:
# ============================================
# Accuracy comparison table
# ============================================

metric_cols = ["R2", "Pearson_r", "RMSE", "MAE"]

accuracy_df = metrics_df.copy()

accuracy_df["model"] = accuracy_df["model"].replace({
    "RandomForest": "Random Forest",
})

accuracy_df = accuracy_df[
    ["species", "model", "dataset"] + metric_cols
].copy()

accuracy_df = accuracy_df.sort_values(
    ["species", "dataset", "model"]
).reset_index(drop=True)

display(accuracy_df)

,species,model,dataset,R2,Pearson_r,RMSE,MAE
0,AT21,LightGBM,Test,0.298263,0.548332,0.842763,0.643657
1,AT21,Random Forest,Test,0.299311,0.550316,0.842133,0.641463
2,AT21,LightGBM,Training,0.556015,0.771511,0.666322,0.511277
3,AT21,Random Forest,Training,0.591905,0.802847,0.638823,0.489965
4,NB21,LightGBM,Test,0.291868,0.542054,0.857198,0.649158
5,NB21,Random Forest,Test,0.296611,0.546186,0.854323,0.645024
6,NB21,LightGBM,Training,0.590997,0.789585,0.639533,0.490118
7,NB21,Random Forest,Training,0.679223,0.850194,0.566372,0.437837
8,OS21,LightGBM,Test,0.383270,0.620351,0.768613,0.586068
9,OS21,Random Forest,Test,0.391549,0.627904,0.763437,0.582230
